In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
import platform
import torch
import numpy as np
import pandas as pd

/opt/anaconda3/envs/NLP-bio/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
def parse_pubtator(raw_text):
    documents = {}
    
    for line in raw_text.strip().split('\n'):
        if not line.strip():
            continue
            
        # Parse Title or Abstract lines (separated by pipes '|')
        if '|t|' in line or '|a|' in line:
            parts = line.split('|', 2)
            doc_id = parts[0]
            text_type = parts[1]
            content = parts[2]
            
            if doc_id not in documents:
                # Initialize document dictionary
                documents[doc_id] = {'title': '', 'abstract': '', 'entities': []}
            
            if text_type == 't':
                documents[doc_id]['title'] = content
            elif text_type == 'a':
                documents[doc_id]['abstract'] = content
                
        # Parse Annotation lines (separated by tabs '\t')
        elif '\t' in line:
            parts = line.split('\t')
            doc_id = parts[0]
            
            # Ensure document exists (handles cases where ID is slightly mismatched, e.g., 763772 vs 25763772)
            if doc_id not in documents:
                documents[doc_id] = {'title': '', 'abstract': '', 'entities': []}
                
            start_char = int(parts[1])
            end_char = int(parts[2])
            mention = parts[3]
            tag = parts[4]  # Your Txxx tags (e.g., T116,T123)
            
            documents[doc_id]['entities'].append({
                'start': start_char,
                'end': end_char,
                'mention': mention,
                'tag': tag
            })

    # Consolidate title and abstract for accurate character offsets
    for doc_id, doc_data in documents.items():
        # PubTator offsets are calculated based on "Title + [space] + Abstract"
        full_text = doc_data['title']
        if doc_data['abstract']:
            full_text += " " + doc_data['abstract']
            
        doc_data['full_text'] = full_text

    return documents

# Example usage with your data
raw_data = """25763772|t|DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis
25763772|a|Pseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease...
25763772	0	5	DCTN4	T116,T123	C4308010
25763772	23	63	chronic Pseudomonas aeruginosa infection	T047	C0854135"""

parsed_docs = parse_pubtator(raw_data)

In [2]:
import spacy
from spacy.training import Example

# Load a blank English model for tokenization
nlp = spacy.blank("en") 

def create_spacy_training_data(parsed_docs):
    training_data = []
    
    for doc_id, doc_data in parsed_docs.items():
        text = doc_data['full_text']
        entities = []
        
        for ent in doc_data['entities']:
            # Some mentions might have multiple tags (e.g., "T116,T123"). 
            # We'll split them and just use the first one for simplicity, 
            # or you can create a composite label.
            primary_tag = ent['tag'].split(',')[0] 
            
            # Append tuple: (start_offset, end_offset, label)
            entities.append((ent['start'], ent['end'], primary_tag))
            
        training_data.append((text, {"entities": entities}))
        
    return training_data

spacy_data = create_spacy_training_data(parsed_docs)

# Verify alignment
for text, annotations in spacy_data:
    doc = nlp.make_doc(text)
    try:
        example = Example.from_dict(doc, annotations)
        print(f"Successfully aligned entities for document length {len(text)}")
    except ValueError as e:
        print(f"Alignment error: {e}")

/opt/anaconda3/envs/NLP-bio/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Successfully aligned entities for document length 209


{'entities': [(0, 5, 'T116'), (23, 63, 'T047')]}

In [ ]:
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

label_list = []
label2id =  {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    label2id = label2id,
    id2label = id2label,

)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") #might have issues on macbook.
else:
    device = torch.device("cpu")
model.to(device)


Loading tokenizer...
Loading model to CPU...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 76476.68it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Moving to MPS (Mac GPU)...
Success! Model is on Mac GPU.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy = "epoch",
    per_device_train_batch_size = 16,
    num_train_epochs = 3,
    use_mps_device=True if device.type == "mps" else False
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset, #need training set
    eval_dataset = dev_dataset, #need dev data set
    tokenizer = tokenizer
)

In [ ]:
NER_Pipeline = pipeline(
    "ner",
    model = model,
    tokenizer=tokenizer,
    device = device 
    )